# FairTrip

**Team:** Likhita Nallapati, Shriya Rawal, Michael Huang  
**Course:** CS 614  
**File:** fairtrip_philly_reviews.csv


In [ ]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
base_path = '/content/drive/MyDrive/FairTrip_Project/yelp folder'

review_path    = os.path.join(base_path, 'yelp_academic_dataset_review.json')
business_path  = os.path.join(base_path, 'yelp_academic_dataset_business.json')
user_path      = os.path.join(base_path, 'yelp_academic_dataset_user.json')
checkin_path   = os.path.join(base_path, 'yelp_academic_dataset_checkin.json')
tip_path       = os.path.join(base_path, 'yelp_academic_dataset_tip.json')

In [ ]:
def load_yelp_file(path, nrows=None):
    data = []
    with open(path, 'r') as f:
        for i, line in enumerate(f):
            if nrows and i >= nrows:
                break
            data.append(json.loads(line))
    return pd.DataFrame(data)

# Load businesses
df_business = load_yelp_file(business_path)

# For reviews, loading sample
df_reviews_sample = load_yelp_file(review_path, nrows=100000)

In [ ]:
# By City
df_philly = df_business[df_business['city'] == 'Philadelphia']
philly_ids = set(df_philly['business_id'])

# Load respective city reviews
philly_reviews = []
with open(review_path, 'r') as f:
    for line in f:
        record = json.loads(line)
        if record['business_id'] in philly_ids:
            philly_reviews.append(record)

df_reviews = pd.DataFrame(philly_reviews)

In [ ]:
print(df_reviews.head(5))

                review_id                 user_id             business_id  \
0  BiTunyQ73aT9WBnpR9DZGw  OyoGAe7OKpv6SyGZT5g77Q  7ATYjTIgM3jUlt4UM3IypQ   
1  AqPFMleE6RsU23_auESxiA  _7bHUi9Uuf5__HHc_Q8guQ  kxX2SOes4o-D3ZQBkiMRfA   
2  JrIxlS1TzJ-iCu79ul40cQ  eUta8W_HdHMXPzLBBZhL1A  04UD14gamNjLY0IDYVhHJg   
3  8JFGBuHMoiNDyfcxuWNtrA  smOvOajNG0lS4Pq7d8g4JQ  RZtGWDLCAtuipwaZ-UfjmQ   
4  oyaMhzBSwfGgemSGuZCdwQ  Dd1jQj7S-BFGqRbApFzCFw  YtSqYv1Q_pOltsVPSx54SA   

   stars  useful  funny  cool  \
0    5.0       1      0     1   
1    5.0       1      0     1   
2    1.0       1      2     1   
3    4.0       0      0     0   
4    5.0       0      0     0   

                                                text                 date  
0  I've taken a lot of spin classes over the year...  2012-01-03 15:28:18  
1  Wow!  Yummy, different,  delicious.   Our favo...  2015-01-04 00:01:03  
2  I am a long term frequent customer of this est...  2015-09-23 23:10:31  
3  Good food--loved the gnocchi wi

In [ ]:
#sanity check for Philly
df_check = df_reviews.merge(
    df_business[['business_id', 'name', 'city', 'categories']],
    on='business_id',
    how='left'
)

print(df_check[['business_id', 'name', 'city', 'categories', 'stars', 'text']].head(10))

              business_id                                               name  \
0  7ATYjTIgM3jUlt4UM3IypQ                         Body Cycle Spinning Studio   
1  kxX2SOes4o-D3ZQBkiMRfA                                              Zaika   
2  04UD14gamNjLY0IDYVhHJg                                           Dmitri's   
3  RZtGWDLCAtuipwaZ-UfjmQ                                          LaScala's   
4  YtSqYv1Q_pOltsVPSx54SA                                  Rittenhouse Grill   
5  eFvzHawVJofxSnD7TgbZtg                                    Good Karma Cafe   
6  rjuWz_AD3WfXJc03AhIO_w                                        The N Crowd   
7  kq5Ghhh14r-eCxlVmlyd8w                                  The Coventry Deli   
8  oBhJuukGRqPVvYBfTkhuZA                                        Square 1682   
9  jTI5Xjk27An8ceJ6VwpXiQ  DoubleTree by Hilton Hotel Philadelphia Center...   

           city                                         categories  stars  \
0  Philadelphia  Active Life, Cycling Clas

In [ ]:
#positive & negative

In [ ]:
positive_keywords = [
    "excellent","amazing","friendly","fast service","clean",
    "delicious","highly recommend","great atmosphere","professional","worth it"
]

negative_keywords = [
    "rude","slow service","dirty","overpriced","disappointing",
    "poor quality","bad experience","unprofessional","never again","long wait"
]

In [ ]:
import re

pos_pattern = "|".join(positive_keywords)
neg_pattern = "|".join(negative_keywords)

In [ ]:
df_check['has_positive'] = df_check['text'].str.contains(pos_pattern, case=False, na=False)
df_check['has_negative'] = df_check['text'].str.contains(neg_pattern, case=False, na=False)

In [ ]:
# Reviews with any positive keywords
df_check[df_check['has_positive']]

# Reviews with any negative keywords
df_check[df_check['has_negative']]

,review_id,user_id,business_id,stars,useful,funny,cool,text,date,name,city,categories,has_positive,has_negative
8,YcLXh-3UC9y6YFAI9xxzPQ,G0DHgkSsDozqUPWtlxVEMw,oBhJuukGRqPVvYBfTkhuZA,4.0,0,0,0,The only reason I didn't give this restaurant ...,2015-03-05 03:37:54,Square 1682,Philadelphia,"American (New), Breakfast & Brunch, Bars, Nigh...",True,True
25,mSggForidMf0eUaK-Qnnvg,x-LrGPXN7WFX15Qk9B6_YQ,eMiN8nm70jjKg8izikVWDA,4.0,0,1,0,"Oh Chickie's and Pete's, you are the perfect p...",2012-07-16 13:25:18,Chickie's & Pete's,Philadelphia,"Seafood, Nightlife, Sports Bars, Bars, Restaur...",False,True
26,HME_ksGph3se7Aze5hxa-Q,kSMOJwJXuEUqzfmuFncK4A,kxX2SOes4o-D3ZQBkiMRfA,2.0,0,0,1,Dine-in gets 2 stars. Disappointing service & ...,2014-07-13 17:25:47,Zaika,Philadelphia,"Halal, Pakistani, Restaurants, Indian",False,True
35,4VBh_hoFiDly0vJffgV_JQ,XLd5uU0OLOnogy_kefXslw,fjAbzsW03bW3EvC4-e184g,5.0,1,0,1,I was a little worried about seeing the review...,2016-01-04 14:33:54,Penn Dermatology,Philadelphia,"Dermatologists, Doctors, Health & Medical",False,True
68,nCdhMSQA0apDuB_oho5ang,2dyfZNhtyuHdestczTgWjQ,Dv6RfXLYe1atjgz3Xf4GGw,4.0,0,0,2,Super lunch option\nWatch out for the yoga mat...,2012-08-15 19:17:49,HipCityVeg,Philadelphia,"Burgers, Vegetarian, Restaurants, Vegan",True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
967492,QYxM3kbnWakqhZRy8dIf1g,cJR2dSsbg7Z8zDklnan-Cw,Yra4YGXmiHv7My5MZ8fjOw,1.0,19,2,0,Worst experience ever! Be prepared to be yell...,2012-09-01 06:18:51,Affordable Electric,Philadelphia,"Home Services, Electricians",False,True
967507,DRHHREVvbY8IjiwAdpzIXw,6DrKoAxWowKnJEG6alpHhg,JHAfVAOo51rxW0Hh9XGxpw,1.0,16,0,0,Wow. Where do I begin?\nLets start at the top....,2016-12-05 01:13:19,Lovely Bride,Philadelphia,"Accessories, Bridal, Fashion, Shopping",False,True
967521,1_ulO0tLglmyYQfuXjPQUg,sRayUe6Y_YSXOK7pKfdlBg,IzLOX3tUTqQFBb9kHdu_wA,3.0,9,0,2,"Tons of top-roping, lead routes, and space... ...",2011-01-27 16:49:38,Go Vertical,Philadelphia,"Climbing, Active Life, Fitness & Instruction, ...",True,True
967540,ZcEx4UEnTnR_TEPEqwkKjA,gkg9VqsxPCgpfYXO1dl8CA,Ea663rIHyKXz2VP2DPH7Cg,4.0,3,0,0,I decided to try this place out after Christma...,2020-01-13 04:21:38,Tinsel,Philadelphia,"Pop-Up Restaurants, Nightlife, Bars, Restaurants",True,True


In [ ]:
# Businesses + Check-ins
# Each row in the check-in file stores all check-in timestamps for one business as a single comma-separated string - we convert that into a `checkin_count` integer per business

df_philly_biz = df_business[df_business['city'] == 'Philadelphia'].copy()

df_checkin = load_yelp_file(checkin_path)
df_checkin['checkin_count'] = df_checkin['date'].str.split(', ').apply(len)
df_checkin_counts = df_checkin[['business_id', 'checkin_count']]

df_philly_biz = df_philly_biz.merge(df_checkin_counts, on='business_id', how='left')
df_philly_biz['checkin_count'] = df_philly_biz['checkin_count'].fillna(0).astype(int)

print(f'Philly businesses: {len(df_philly_biz):,}')
print(df_philly_biz[['business_id', 'name', 'review_count', 'checkin_count']].head())

Philly businesses: 14,569
              business_id                name  review_count  checkin_count
0  MTSW4McQd7CbVtyjqoe9mw  St Honore Pastries            80            335
1  MUTTqe8uqyMdBl186RmNeA            Tuna Bar           245            172
2  ROeacJQwBeh05Rqg7F6TCg                 BAP           205            221
3  QdN72BWoyFypdGJhhI5r7g             Bar One            65             81
4  Mjboz24M9NlBeiOJKLEd_Q    DeSandro on Main            41              9


In [ ]:
# Users (only those who reviewed a Philly business)
# The user file is large (~2M rows), so we stream it line by line and keep only users whose user_id appears in the Philly review table

philly_user_ids = set(df_reviews['user_id'])
print(f'Distinct Philly reviewers: {len(philly_user_ids):,}')

philly_users = []
with open(user_path, 'r') as f:
    for line in f:
        record = json.loads(line)
        if record['user_id'] in philly_user_ids:
            philly_users.append(record)

df_users = pd.DataFrame(philly_users)
print(f'Philly users loaded: {len(df_users):,}')
print(df_users.head())

Distinct Philly reviewers: 279,857
Philly users loaded: 279,855
                  user_id    name  review_count        yelping_since  useful  \
0  qVc8ODYU5SZjKXVBgXdI7w  Walker           585  2007-01-25 16:47:26    7217   
1  j14WgRoU_-2ZE1aw1dXrJg  Daniel          4333  2009-01-25 04:35:42   43091   
2  q_QQ5kBBwlCcbL1s4NVK3g    Jane          1221  2005-03-14 20:26:35   14953   
3  AUi8MPWJ0mLkMfwbui27lg    John           109  2010-01-07 18:32:04     154   
4  1McG5Rn_UDkmlkZOrsdptg  Teresa             7  2009-05-26 16:11:11      18   

   funny   cool                                              elite  \
0   1259   5994                                               2007   
1  13066  27281  2009,2010,2011,2012,2013,2014,2015,2016,2017,2...   
2   9940  11211       2006,2007,2008,2009,2010,2011,2012,2013,2014   
3     20     23                                                      
4      3     13                                                      

                                  

In [ ]:
# Tips (on Philly businesses)
# Tips are short user-written suggestions, separate from reviews and without star ratings. We keep only tips left on Philadelphia businesses

df_tip = load_yelp_file(tip_path)
df_philly_tips = df_tip[df_tip['business_id'].isin(philly_ids)].copy()

print(f'Philly tips: {len(df_philly_tips):,}')
print(df_philly_tips.head())

Philly tips: 118,546
                   user_id             business_id  \
2   -copOvldyKh1qr-vzkDEvw  MYoRNLb5chwjQe3c_k37Gg   
20  FQ-zmWPEG_pjSQx6pt3Efw  3ZynJ94VpIdDlaArmEp2Rg   
34  YnlCpuaBa3qWBp4te8pGmA  XIKYdKWq72zUYsq8NBxcCQ   
48  Rr4cLb6Go91FT134o6RsKg  eMiN8nm70jjKg8izikVWDA   
53  fJhr0G2JBNkfqpbIwkEQHg  eJ77e9lGxY3ArzaoDbHhYw   

                                      text                 date  \
2   It's open even when you think it isn't  2013-08-18 00:56:08   
20  Yes, I'm eating here again. Breakfast!  2012-10-12 15:16:13   
34     The honey glazed salmon is amazing!  2018-01-14 15:00:01   
48                   Mmm Yummy Crab Fries!  2011-10-16 23:43:10   
53                Good specials, nice menu  2013-02-16 21:34:37   

    compliment_count  
2                  0  
20                 0  
34                 0  
48                 0  
53                 0  


# Merge and download

In [ ]:
# Tip counts per business
tip_counts = df_philly_tips.groupby('business_id')['text'].count().reset_index()
tip_counts.columns = ['business_id', 'tip_count']

# Merge everything
df_final = df_check.merge(
    df_philly_biz[['business_id', 'review_count', 'checkin_count']].rename(
        columns={'review_count': 'business_review_count'}),
    on='business_id', how='left'
).merge(
    df_users[['user_id', 'review_count', 'average_stars']].rename(columns={
        'review_count': 'user_review_count',
        'average_stars': 'user_avg_stars'
    }),
    on='user_id', how='left'
).merge(
    tip_counts, on='business_id', how='left'
)

df_final['tip_count'] = df_final['tip_count'].fillna(0).astype(int)

# Filter: users with ≥ 5 reviews
user_counts = df_final.groupby('user_id')['review_id'].count()
active_users = user_counts[user_counts >= 5].index
df_final = df_final[df_final['user_id'].isin(active_users)]

# Drop duplicates
df_final = df_final.drop_duplicates(subset=['user_id', 'business_id'], keep='last')

# Final columns
final_cols = [
    'review_id', 'user_id', 'business_id', 'stars',
    'useful', 'funny', 'cool', 'text', 'date',
    'name', 'city', 'categories',
    'business_review_count', 'checkin_count', 'tip_count',
    'user_review_count', 'user_avg_stars',
    'has_positive', 'has_negative'
]
df_final = df_final[final_cols]

# Sanity checks
print(f"Total reviews:      {len(df_final):,}")
print(f"Unique users:       {df_final['user_id'].nunique():,}")
print(f"Unique businesses:  {df_final['business_id'].nunique():,}")
print(f"Has_positive:       {df_final['has_positive'].sum():,}")
print(f"Has_negative:       {df_final['has_negative'].sum():,}")
print(f"\n{df_final.dtypes}")

Total reviews:      578,777
Unique users:       37,666
Unique businesses:  14,344
Has_positive:       264,670
Has_negative:       40,359

review_id                 object
user_id                   object
business_id               object
stars                    float64
useful                     int64
funny                      int64
cool                       int64
text                      object
date                      object
name                      object
city                      object
categories                object
business_review_count      int64
checkin_count              int64
tip_count                  int64
user_review_count        float64
user_avg_stars           float64
has_positive                bool
has_negative                bool
dtype: object


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Download
from google.colab import files
df_final.to_csv('fairtrip_philly_reviews.csv', index=False)
files.download('fairtrip_philly_reviews.csv')